> **Superseded (2026-11-17).** This notebook ran under the earlier RQ1
> protocol: presence-vs-background Random Forest (background as
> pseudo-absence), ROC-AUC / Boyce, random block holdouts. The RQ1
> classifier is now the presence-only logistic regression, evaluated by
> sensitivity at a fixed area fraction under leave-one-block-out
> ([`thesis/notes.md`](../../thesis/notes.md#rq1-presence-only),
> [`presence_only_method_selection.ipynb`](presence_only_method_selection.ipynb)).
> Its model results are not reported; the label-purity counts are facts of the data and still stand.

# RQ1 — label purity of citizen-science presences

**Purpose.** A 10 m pixel at an iNaturalist point may not actually be
dominated by *Acacia dealbata* — GPS error, or an isolated tree logged
in the middle of other vegetation. If this happens often, it adds
label noise to every citizen-science result in RQ1, including the flat
effort-reduction curve
(`docs/research_questions.md#rq1--citizen-science-as-a-substitute-for-expert-data`).
This is pipeline step 6, "label purity", listed as open in
`docs/research_questions.md`.

**Two cheap checks, both asked for directly:**
1. Where a citizen-science point lands near an expert polygon (already
   known ground truth, independent of iNaturalist — see
   `expert_polygons.ipynb`), what fraction is actually *inside* the
   polygon rather than just nearby?
2. In TESSERA embedding space, which presences sit far from the rest
   of the *Acacia* presence cloud? An isolated or misidentified tree
   should look different from a typical stand.

Both checks only flag *candidates* for noise, not confirmed errors —
there is no independent per-point ground truth for most of the
2 465 presences. Check 2 also relates to the identification-error risk
discussed earlier (candidate explanation 5, not pursued as its own
investigation because it would require abandoning the curve).


> **Question numbering (note of 2026-11-13).** Current numbering in
> `docs/research_questions.md`: RQ1 = effort-reduction curve (citizen
> science against expert data); RQ2 = from points to reference data
> (segmentation mask, grade and accuracy filters); RQ3 =
> transferability away from the sampled area; RQ4 = TESSERA against
> Sentinel-2.
> In this notebook "RQ1" is the effort-reduction curve, which is still
> RQ1.

## 1. Presences, expert polygons and embeddings

Same files as the other RQ1 notebooks: the cleaned citizen-science
presences (`inat_dealbata_clean.csv`), the cleaned expert polygons
(`expert_polygons_clean.gpkg`, `classifica = 1` only), and the RQ1
pilot's TESSERA embeddings. The citizen-science → pixel match is done
by hand here (not `src.embeddings.match_to_pixel`) because this
notebook needs to keep each presence's own attributes (observer,
positional accuracy) attached to its embedding, which that shared
helper does not return.


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.embeddings import load_pilot_embeddings

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"
pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)

dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
pixel_to_idx = {p: i for i, p in enumerate(pilot_df["pixel"])}
dealbata["pilot_idx"] = dealbata["pixel"].map(pixel_to_idx)
matched = dealbata.dropna(subset=["pilot_idx"]).copy()
matched["pilot_idx"] = matched["pilot_idx"].astype(int)
emb_all = pilot_emb[matched["pilot_idx"].values]
valid = ~np.isnan(emb_all).any(axis=1)
matched = matched[valid].reset_index(drop=True)
emb = emb_all[valid]
matched["x"] = pilot_df["x"].values[matched["pilot_idx"].values]
matched["y"] = pilot_df["y"].values[matched["pilot_idx"].values]

expert = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg").to_crs(32629)
expert_union = expert.union_all()

print(f"presences matched to a pilot pixel with a valid embedding: {len(matched):,}/{len(dealbata):,}")


## 2. Check 1 — distance to the nearest expert polygon

For every citizen-science presence, distance to the nearest expert
*A. dealbata* polygon (0 if the point falls inside one). This only
says something for the minority of presences that have an expert
polygon nearby at all — most of the pilot has no expert coverage, so
"far from any polygon" is not evidence of anything. The point of this
check is the presences that *are* close: if citizen science and
experts are describing the same stand, the point should mostly land
inside it, not just near it.


In [ ]:
pres_pts = gpd.GeoDataFrame(matched, geometry=gpd.points_from_xy(matched["x"], matched["y"]), crs=32629)
pres_pts["dist_to_expert_m"] = pres_pts.distance(expert_union)

bins = [-0.1, 0, 50, 100, 500, 5000, np.inf]
labels = ["inside", "0-50 m", "50-100 m", "100-500 m", "500 m-5 km", "> 5 km"]
pres_pts["expert_distance_band"] = pd.cut(pres_pts["dist_to_expert_m"], bins=bins, labels=labels)
pres_pts["expert_distance_band"].value_counts().reindex(labels)


In [ ]:
near = pres_pts[pres_pts["dist_to_expert_m"] <= 100]
print(f"presences within 100 m of an expert polygon: {len(near)}")
print(f"of those, actually inside one: {(near['dist_to_expert_m'] == 0).sum()} "
      f"({100 * (near['dist_to_expert_m'] == 0).mean():.1f}%)")
near[["observer", "dist_to_expert_m"]].assign(inside=near["dist_to_expert_m"] == 0).groupby("observer")["inside"].agg(["size", "mean"])


**Reading.** Only 21 of 2,464 presences (0.9%) land literally inside
an expert polygon — close to the 22 known before pixel-level dedup
(`expert_polygons.ipynb`). Widening to "within 100 m" picks up 262
presences, but only 8.0% of those are actually inside one; 92% sit
nearby but outside the mapped polygon. Observer A and other observers
confirm at a similar rate (8.0% vs. 9.1%), so this is not an
Observer-A-specific problem. On its own this check is inconclusive:
expert polygons were digitized as specific stands/segments, many very
small (section 3 of `expert_polygons.ipynb`: 1,353 are <100 m²
segmentation tiles), so a point 20-80 m away plausibly describes the
same real-world occurrence the expert just didn't trace to that exact
pixel — not necessarily a GPS or identification error. It also only
covers 262 of 2,464 presences (11%); it cannot characterise purity for
the rest of the pilot, which has no expert coverage nearby at all.


## 3. Check 2 — embedding-space outliers

For each presence, the mean cosine distance to its 10 nearest
neighbours among the *other* presences. A presence whose neighbours
are all far away (in embedding terms) looks unlike the rest of the
*Acacia* class — a candidate for GPS error, misidentification, or a
genuinely atypical pixel (isolated tree, mixed pixel at a stand edge).
This covers every matched presence, not just the ones near expert
polygons.


In [ ]:
from sklearn.neighbors import NearestNeighbors

K = 10
nn = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(emb)
dist, _ = nn.kneighbors(emb)
matched["knn_dist"] = dist[:, 1:].mean(axis=1)  # column 0 is the point itself (distance 0)

matched["knn_dist"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])


In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE = "#2a78d6", "#eb6834"
p95 = matched["knn_dist"].quantile(0.95)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(matched["knn_dist"], bins=60, color=BLUE, alpha=0.85)
ax.axvline(p95, color=ORANGE, lw=1.5, ls="--", label=f"95th pctl ({p95:.3f})")
ax.set(xlabel="mean cosine distance to 10 nearest other presences", ylabel="presences",
       title="Embedding-space outlierness of citizen-science presences")
ax.legend()
plt.tight_layout()
plt.savefig("/tmp/label_purity_knn_hist.png", dpi=150)
plt.show()


In [ ]:
outliers = matched[matched["knn_dist"] >= p95].copy()
rest = matched[matched["knn_dist"] < p95]
print(f"flagged as outliers (top 5% by knn_dist): {len(outliers)}")
print()
print("positional_accuracy (m), median — outliers vs. rest:",
      outliers["positional_accuracy"].median(), "vs.", rest["positional_accuracy"].median())
print("accuracy_missing share — outliers vs. rest:",
      round(outliers["accuracy_missing"].mean(), 3), "vs.", round(rest["accuracy_missing"].mean(), 3))
print()
print("observer share among outliers:")
print(outliers["observer"].value_counts(normalize=True).round(3).head())
print()
print("observer share among the rest:")
print(rest["observer"].value_counts(normalize=True).round(3).head())


### Cross-check against check 1

If `knn_dist` is a meaningful purity proxy, the presences already
confirmed by an expert polygon (distance 0 in check 1) should not be
disproportionately flagged as outliers here — they are independently
known to be real, in-stand *Acacia*.


In [ ]:
confirmed_idx = pres_pts.index[pres_pts["dist_to_expert_m"] == 0]
confirmed = matched.loc[matched.index.isin(confirmed_idx)]
print(f"knn_dist, confirmed-inside-expert-polygon presences (n={len(confirmed)}): "
      f"median {confirmed['knn_dist'].median():.4f}")
print(f"knn_dist, all other presences (n={len(matched) - len(confirmed)}): "
      f"median {matched.loc[~matched.index.isin(confirmed_idx), 'knn_dist'].median():.4f}")
print(f"share of confirmed presences flagged as outliers (>= p95): "
      f"{(confirmed['knn_dist'] >= p95).mean():.3f} (vs. 0.05 expected if no different from the rest)")


**Reading.** The cross-check supports `knn_dist` as a meaningful proxy:
presences independently confirmed inside an expert polygon have a
lower median `knn_dist` (0.055) than the rest (0.092), and **none** of
the 21 confirmed presences are flagged as outliers, against 5%
expected by chance — real, in-stand *Acacia* looks less "outlier-like"
in embedding space than the average presence, as it should if the
proxy is working.

Among the flagged 5% (124 presences): positional accuracy is no worse
than the rest (median 8.0 m vs. 9.0 m), so GPS imprecision alone does
not explain them. The clearest signal is observer composition —
non-dominant observers make up 44% of flagged outliers vs. only 7.8%
of the rest, a roughly 6x over-representation. Observer A's presences
are more homogeneous/typical in embedding space than occasional
observers' are; own reasoning, no literature checked for whether this
generalises beyond this pilot.


## 4. Does it matter? Re-running the 0% (citizen-science-only) point without the flagged outliers

The real question is not whether outliers exist (they always will,
at some rate) but whether they are numerous/extreme enough to move
the result. Same setup as `presence_background_citizen_science.ipynb`
(Random Forest, same shared background pool, same 10 km spatial
blocks, same 5 block-holdout seeds), run twice: once with all
presences, once dropping the top-5% most embedding-outlier presences
flagged above. If the metrics barely move, label impurity at this
rate is not what explains the flat curve.


In [ ]:
BG_FILE = EMB_DIR / "tessera_v1.1_2023_rq1_background" / "background_points.npz"
from src.background import load_background

# Built in notebooks/05_data_integration/background_pools.ipynb; load_background() checks the
# file and its inputs against data/processed_manifest.csv before loading.
bg = load_background(ROOT, "uniform")
bg_x, bg_y, bg_emb = bg["x"], bg["y"], bg["emb"]

from src.embeddings import block_id

pres_block = block_id(matched["x"].values, matched["y"].values)
bg_block = block_id(bg_x, bg_y)
pres_blocks_unique = sorted(set(pres_block) | set(bg_block))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from src.evaluation import continuous_boyce

keep_mask = matched["knn_dist"] < p95  # drop the flagged 5%

# Third variant: the flag is computed inside each seed, from the training presences only.
PER_SEED = "drop top-5% outliers (flag from training presences)"
variants = {"all presences": np.ones(len(matched), dtype=bool), "drop top-5% outliers": keep_mask.values, PER_SEED: None}
purity_results = {name: {"auc": [], "boyce": []} for name in variants}

for seed in range(1, 6):
    rng = np.random.default_rng(seed)
    test_blocks = set(rng.choice(pres_blocks_unique, size=max(1, round(0.2 * len(pres_blocks_unique))), replace=False))
    train_mask_bg = ~np.isin(bg_block, list(test_blocks))
    test_mask_bg = np.isin(bg_block, list(test_blocks))
    for name, keep in variants.items():
        if name == PER_SEED:
            in_train = ~np.isin(pres_block, list(test_blocks))
            d, _ = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(emb[in_train]).kneighbors(emb[in_train])
            knn_train = d[:, 1:].mean(axis=1)
            keep = np.ones(len(matched), dtype=bool)
            keep[np.flatnonzero(in_train)[knn_train >= np.quantile(knn_train, 0.95)]] = False
        # The filter applies to training only: both variants are scored on the same test set.
        train_mask_pres = keep & ~np.isin(pres_block, list(test_blocks))
        test_mask_pres = np.isin(pres_block, list(test_blocks))
        X_train = np.vstack([emb[train_mask_pres], bg_emb[train_mask_bg]])
        y_train = np.r_[np.ones(train_mask_pres.sum()), np.zeros(train_mask_bg.sum())]
        X_test = np.vstack([emb[test_mask_pres], bg_emb[test_mask_bg]])
        y_test = np.r_[np.ones(test_mask_pres.sum()), np.zeros(test_mask_bg.sum())]
        if test_mask_pres.sum() < 5 or test_mask_bg.sum() < 5:
            continue
        clf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42).fit(X_train, y_train)
        proba = clf.predict_proba(X_test)[:, 1]
        purity_results[name]["auc"].append(roc_auc_score(y_test, proba))
        purity_results[name]["boyce"].append(continuous_boyce(proba[y_test == 1], proba[y_test == 0]))

for name in list(variants)[1:]:
    auc_diff = np.array(purity_results[name]["auc"]) - np.array(purity_results["all presences"]["auc"])
    print(f"{name}: paired ROC-AUC difference to all presences, per seed: {np.round(auc_diff, 4)}; "
          f"mean {auc_diff.mean():+.4f}, sd {auc_diff.std(ddof=1):.4f}")

pd.DataFrame({name: {"ROC-AUC": np.mean(r["auc"]), "Boyce": np.mean(r["boyce"]), "n splits": len(r["auc"])}
              for name, r in purity_results.items()}).T


**Reading.** Both variants are scored on the same test set: the
filter applies to training only. Dropping the 124 flagged presences
(5%) from training does not raise ROC-AUC: 0.835 → 0.826, a paired
difference of −0.009 (sd 0.014 across the five seeds), negative in all
five (−0.035 in one seed, about −0.003 in the others). Boyce is 0.981
and 0.983. With the flag computed inside each seed from the training
presences only, the result is the same: 0.824, a paired difference of
−0.012 (sd 0.019), negative in all five seeds (−0.045 in one), Boyce
0.978. Removing these points unlocks no extra headroom. The small
drop may reflect that the test set still holds flagged presences,
which the filtered model never saw (possible explanation, not tested).
Label impurity, at the rate this check can detect, is not what explains
the flat effort-reduction curve or the citizen-science/expert gap seen
elsewhere.


## 5. Conclusion

Neither check found a large-scale label-purity problem. Only a small,
low-power subset of presences (262 of 2,464) can be checked against
independent expert ground truth at all, and that check is inconclusive
by itself (8% confirmed inside a polygon, but "nearby, not inside" is
not necessarily wrong). The embedding-based outlier check is validated
by that same ground truth (confirmed presences score as clearly less
outlier-like than average) and flags a plausible, modest-sized (5%)
group of atypical presences, concentrated among non-dominant observers
— but excluding them from training does not improve the
citizen-science-only result (ROC-AUC 0.009 lower on the same test
set). **Label purity is not the explanation
for the flat curve.** This complements the earlier finding that it is
habitat concentration in the expert arm
(`expert_polygon_geographic_diversity.ipynb`), not a problem with the
citizen-science arm.

**Caveats:**
- Check 1 only validates the small minority of presences near expert
  coverage; it says nothing about purity in the rest of the pilot.
- Check 2 is a proxy (embedding distance), not ground truth — a
  genuinely atypical but correctly identified stand (e.g. a small or
  edge-of-range patch) would also score as an "outlier" here, so the
  flagged 5% is not the same as "5% mislabelled". No human visually
  reviewed the flagged points against imagery.
- The threshold (top 5%) is arbitrary; not tested for sensitivity to
  other cutoffs.
- The outlier flag of Sections 3 and of the second row of Section 4 is
  computed on all presences, including those in the test blocks of
  each seed; the third row of Section 4 computes it from the training
  presences only.
